In [1]:
from __future__ import annotations
import json
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [3]:
RANDOM_STATE = 42

CSV_PATH = Path("data/tl9.csv")
OUTPUT_DIR = Path("fanet_nckh_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = [
    "tx_pps",
    "rx_pps",
    "drop_pps",
    "tx_bps",
    "rx_bps",
    "window_pdr",
    "window_plr",
    "tx_mean_pkt_bytes",
    "rx_mean_pkt_bytes",
    "delay_ms",
    "jitter_ms",
    "speed",
    "neighbors",
]

LEAKAGE_COLUMNS_EXCLUDED = [
    "window_id",
    "time_s",
    "flow_id",
    "srcNode",
    "dstNode",
    "srcIp",
    "dstIp",
    "srcPort",
    "dstPort",
    "x",
    "y",
    "z",
]

REQUIRED_COLUMNS = {
    "time_s",
    "flow_id",
    "txPackets",
    "rxPackets",
    "txBytes",
    "rxBytes",
    "lostPackets",
    "delay_ms",
    "jitter_ms",
    "speed",
    "neighbors",
    "attack_label",
}

In [4]:
def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if pd.isna(value):
        return None
    return value

In [5]:
def load_dataset(csv_path: str | Path) -> pd.DataFrame:
    path = Path(csv_path)

    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy CSV: {path}")

    df = pd.read_csv(path)

    missing = sorted(REQUIRED_COLUMNS - set(df.columns))
    if missing:
        raise ValueError(f"CSV thiếu các cột bắt buộc: {missing}")

    if df.empty:
        raise ValueError("CSV không có dữ liệu.")

    return df

In [6]:
raw_df = load_dataset(CSV_PATH)

print("Kích thước dataset:", raw_df.shape)
print("Số flow:", raw_df["flow_id"].nunique())
print("\nPhân bố nhãn:")
display(raw_df["attack_label"].value_counts())
display(raw_df.head())

Kích thước dataset: (2325182, 34)
Số flow: 42551

Phân bố nhãn:


attack_label
tcp_syn_flood    1169351
normal           1155766
dos_udp_flood         65
Name: count, dtype: int64

,window_id,time_s,flow_id,srcNode,dstNode,tier,srcIp,dstIp,srcPort,dstPort,...,syn_count,ack_count,rst_count,syn_ack_ratio,x,y,z,speed,neighbors,attack_label
0,5,5.0,1,0,17,0,10.1.0.1,10.1.0.18,49153,9000,...,1,0,0,1.0000,286.1577,287.8753,50.0,0.9756,7,normal
1,5,5.0,2,2,17,0,10.1.0.3,10.1.0.18,49153,9002,...,1,0,0,1.0000,237.9460,290.4493,50.0,0.7794,10,normal
2,5,5.0,3,6,17,0,10.1.0.7,10.1.0.18,49153,9006,...,1,0,0,1.0000,259.3936,290.6081,50.0,0.1449,10,normal
3,6,6.0,1,0,17,0,10.1.0.1,10.1.0.18,49153,9000,...,1,32,0,0.0312,286.1892,288.8503,50.0,0.9756,7,normal
4,6,6.0,2,2,17,0,10.1.0.3,10.1.0.18,49153,9002,...,1,31,0,0.0323,238.5353,289.9391,50.0,0.7794,10,normal


In [7]:
def build_window_features(df: pd.DataFrame) -> pd.DataFrame:
    data = df.sort_values(["flow_id", "time_s"]).copy()
    grouped = data.groupby("flow_id", group_keys=False)

    data["dt_s"] = grouped["time_s"].diff()

    cumulative_cols = [
        "txPackets",
        "rxPackets",
        "txBytes",
        "rxBytes",
        "lostPackets",
    ]

    for col in cumulative_cols:
        data[f"d_{col}"] = grouped[col].diff()

    data = data[data["dt_s"] > 0].copy()

    if data.empty:
        raise ValueError(
            "Không tạo được cửa sổ hợp lệ; hãy kiểm tra time_s và flow_id."
        )

    for col in cumulative_cols:
        data[f"d_{col}"] = data[f"d_{col}"].clip(lower=0)

    data["tx_pps"] = data["d_txPackets"] / data["dt_s"]
    data["rx_pps"] = data["d_rxPackets"] / data["dt_s"]
    data["drop_pps"] = data["d_lostPackets"] / data["dt_s"]

    data["tx_bps"] = 8.0 * data["d_txBytes"] / data["dt_s"]
    data["rx_bps"] = 8.0 * data["d_rxBytes"] / data["dt_s"]

    data["window_pdr"] = np.where(
        data["d_txPackets"] > 0,
        data["d_rxPackets"] / data["d_txPackets"],
        0.0,
    )

    data["window_plr"] = np.where(
        data["d_txPackets"] > 0,
        data["d_lostPackets"] / data["d_txPackets"],
        0.0,
    )

    data["tx_mean_pkt_bytes"] = np.where(
        data["d_txPackets"] > 0,
        data["d_txBytes"] / data["d_txPackets"],
        0.0,
    )

    data["rx_mean_pkt_bytes"] = np.where(
        data["d_rxPackets"] > 0,
        data["d_rxBytes"] / data["d_rxPackets"],
        0.0,
    )

    data["traffic_active"] = (
        (data["d_txPackets"] > 0)
        | (data["d_rxPackets"] > 0)
    )

    numeric_columns = data.select_dtypes(include=np.number).columns
    data[numeric_columns] = data[numeric_columns].replace(
        [np.inf, -np.inf],
        np.nan,
    )

    return data

In [8]:
window_df = build_window_features(raw_df)

print("Kích thước sau tạo đặc trưng:", window_df.shape)
display(window_df[FEATURES + ["attack_label", "flow_id", "time_s"]].head())

Kích thước sau tạo đặc trưng: (2282631, 50)


,tx_pps,rx_pps,drop_pps,tx_bps,rx_bps,window_pdr,window_plr,tx_mean_pkt_bytes,rx_mean_pkt_bytes,delay_ms,jitter_ms,speed,neighbors,attack_label,flow_id,time_s
3,32.0,33.0,0.0,140288.0,140736.0,1.03125,0.0,548.0,533.090909,0.3830,0.1323,0.9756,7,normal,1,6.0
11,31.0,31.0,0.0,139872.0,139872.0,1.00000,0.0,564.0,564.000000,0.3225,0.0672,0.9756,7,normal,1,7.0
19,31.0,31.0,0.0,139872.0,139872.0,1.00000,0.0,564.0,564.000000,0.3015,0.0451,0.9756,7,normal,1,8.0
27,31.0,31.0,0.0,139872.0,139872.0,1.00000,0.0,564.0,564.000000,0.2908,0.0339,0.9756,7,normal,1,9.0
35,32.0,32.0,0.0,144384.0,144384.0,1.00000,0.0,564.0,564.000000,0.2842,0.0270,0.9756,7,normal,1,10.0


In [9]:
def audit_dataset(
    raw: pd.DataFrame,
    window_data: pd.DataFrame,
) -> dict[str, Any]:

    flow_summary = (
        raw.groupby("attack_label")
        .agg(
            rows=("attack_label", "size"),
            flows=("flow_id", "nunique"),
        )
        .reset_index()
        .to_dict(orient="records")
    )

    leakage_candidates = [
        "flow_id",
        "srcNode",
        "dstNode",
        "srcIp",
        "dstIp",
        "srcPort",
        "dstPort",
    ]

    leakage = []

    for col in leakage_candidates:
        if col not in raw.columns:
            continue

        labels_per_value = raw.groupby(col)["attack_label"].nunique()

        leakage.append(
            {
                "column": col,
                "unique_values": int(raw[col].nunique()),
                "values_linked_to_one_label": int(
                    (labels_per_value == 1).sum()
                ),
                "ratio_one_label": float(
                    (labels_per_value == 1).mean()
                ),
            }
        )

    activity = (
        window_data.groupby("attack_label")
        .agg(
            windows=("attack_label", "size"),
            active_windows=("traffic_active", "sum"),
        )
        .reset_index()
    )

    activity["active_ratio"] = (
        activity["active_windows"] / activity["windows"]
    )

    strict_labels = np.where(
        (window_data["attack_label"] != "normal")
        & (~window_data["traffic_active"]),
        "normal",
        window_data["attack_label"],
    )

    strict_counts = (
        pd.Series(strict_labels)
        .value_counts()
        .to_dict()
    )

    return {
        "shape": [int(raw.shape[0]), int(raw.shape[1])],
        "flow_summary": flow_summary,
        "leakage_audit": leakage,
        "activity_audit": activity.to_dict(orient="records"),
        "strict_active_label_counts": {
            str(k): int(v)
            for k, v in strict_counts.items()
        },
    }

In [10]:
audit = audit_dataset(raw_df, window_df)

print("Tóm tắt số dòng và số flow theo lớp:")
display(pd.DataFrame(audit["flow_summary"]))

print("Kiểm tra cột có nguy cơ leakage:")
display(pd.DataFrame(audit["leakage_audit"]))

print("Tỷ lệ cửa sổ có traffic thực sự hoạt động:")
display(pd.DataFrame(audit["activity_audit"]))

Tóm tắt số dòng và số flow theo lớp:


,attack_label,rows,flows
0,dos_udp_flood,65,1
1,normal,1155766,21179
2,tcp_syn_flood,1169351,21371


Kiểm tra cột có nguy cơ leakage:


,column,unique_values,values_linked_to_one_label,ratio_one_label
0,flow_id,42551,42551,1.000000
1,srcNode,9,9,1.000000
2,dstNode,5,4,0.800000
3,srcIp,9,9,1.000000
4,dstIp,5,4,0.800000
5,srcPort,21376,21375,0.999953
6,dstPort,21177,21176,0.999953


Tỷ lệ cửa sổ có traffic thực sự hoạt động:


,attack_label,windows,active_windows,active_ratio
0,dos_udp_flood,64,6,0.093750
1,normal,1134587,7306,0.006439
2,tcp_syn_flood,1147980,6216,0.005415


In [11]:
def temporal_split_by_flow(
    df: pd.DataFrame,
    train_ratio: float = 0.60,
    val_ratio: float = 0.20,
    purge_windows: int = 1,
) -> pd.DataFrame:

    if not 0 < train_ratio < 1:
        raise ValueError("train_ratio phải nằm trong (0, 1).")

    if (
        not 0 < val_ratio < 1
        or train_ratio + val_ratio >= 1
    ):
        raise ValueError("val_ratio không hợp lệ.")

    parts: list[pd.DataFrame] = []

    for _, group in df.groupby("flow_id"):
        group = group.sort_values("time_s").copy()
        n = len(group)

        if n < 5:
            raise ValueError(
                "Mỗi flow cần ít nhất 5 cửa sổ để chia temporal split."
            )

        train_end = max(1, int(n * train_ratio))
        val_end = max(
            train_end + 1,
            int(n * (train_ratio + val_ratio)),
        )
        val_end = min(val_end, n - 1)

        split = np.full(n, "test", dtype=object)
        split[:train_end] = "train"
        split[train_end:val_end] = "validation"

        purge_idx: set[int] = set()

        for boundary in (train_end, val_end):
            for offset in range(
                -purge_windows,
                purge_windows + 1,
            ):
                idx = boundary + offset

                if 0 <= idx < n:
                    purge_idx.add(idx)

        split[list(purge_idx)] = "purged"

        group["split"] = split
        parts.append(group)

    result = pd.concat(parts, ignore_index=True)

    required_labels = set(
        result["attack_label"].unique()
    )

    for split_name in ("train", "validation", "test"):
        labels = set(
            result.loc[
                result["split"] == split_name,
                "attack_label",
            ]
        )

        missing = sorted(required_labels - labels)

        if missing:
            raise RuntimeError(
                f"Tập {split_name} thiếu lớp: {missing}"
            )

    return result

In [12]:
split_df = temporal_split_by_flow(
    window_df,
    train_ratio=0.60,
    val_ratio=0.20,
    purge_windows=1,
)

split_counts = pd.crosstab(
    split_df["split"],
    split_df["attack_label"],
)

display(split_counts)

attack_label,dos_udp_flood,normal,tcp_syn_flood
split,,,
purged,6,127074,128226
test,11,192767,195269
train,37,651269,658972
validation,10,163477,165513


In [13]:
train_df = split_df[
    split_df["split"] == "train"
].copy()

val_df = split_df[
    split_df["split"] == "validation"
].copy()

test_df = split_df[
    split_df["split"] == "test"
].copy()

X_train = train_df[FEATURES]
y_train = train_df["attack_label"]

X_val = val_df[FEATURES]
y_val = val_df["attack_label"]

X_test = test_df[FEATURES]
y_test = test_df["attack_label"]

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (1310278, 13)
Validation: (329000, 13)
Test: (388047, 13)


In [14]:
def make_candidates(
    random_state: int = RANDOM_STATE,
) -> dict[str, Pipeline]:

    def tree_pipeline(estimator: Any) -> Pipeline:
        return Pipeline(
            [
                (
                    "imputer",
                    SimpleImputer(strategy="median"),
                ),
                ("model", estimator),
            ]
        )

    return {
        "Dummy-most-frequent": Pipeline(
            [
                (
                    "imputer",
                    SimpleImputer(strategy="median"),
                ),
                (
                    "model",
                    DummyClassifier(
                        strategy="most_frequent"
                    ),
                ),
            ]
        ),

        "LogisticRegression": Pipeline(
            [
                (
                    "imputer",
                    SimpleImputer(strategy="median"),
                ),
                ("scaler", StandardScaler()),
                (
                    "model",
                    LogisticRegression(
                        max_iter=3000,
                        class_weight="balanced",
                        random_state=random_state,
                    ),
                ),
            ]
        ),

        "DecisionTree": tree_pipeline(
            DecisionTreeClassifier(
                max_depth=8,
                min_samples_leaf=5,
                class_weight="balanced",
                random_state=random_state,
            )
        ),

        "RandomForest": tree_pipeline(
            RandomForestClassifier(
                n_estimators=500,
                max_depth=12,
                min_samples_leaf=3,
                max_features="sqrt",
                class_weight="balanced_subsample",
                n_jobs=1,
                random_state=random_state,
            )
        ),

        "ExtraTrees": tree_pipeline(
            ExtraTreesClassifier(
                n_estimators=500,
                max_depth=12,
                min_samples_leaf=3,
                max_features="sqrt",
                class_weight="balanced",
                n_jobs=1,
                random_state=random_state,
            )
        ),
    }

In [15]:
def score_predictions(
    y_true: pd.Series,
    y_pred: np.ndarray,
) -> dict[str, float]:

    return {
        "accuracy": float(
            accuracy_score(y_true, y_pred)
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(y_true, y_pred)
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                average="macro",
            )
        ),
        "weighted_f1": float(
            f1_score(
                y_true,
                y_pred,
                average="weighted",
            )
        ),
    }

In [16]:
def evaluate_candidates(
    train_data: pd.DataFrame,
    validation_data: pd.DataFrame,
    candidates: dict[str, Pipeline],
) -> pd.DataFrame:

    rows: list[dict[str, Any]] = []

    X_train_local = train_data[FEATURES]
    y_train_local = train_data["attack_label"]

    X_val_local = validation_data[FEATURES]
    y_val_local = validation_data["attack_label"]

    for name, model in candidates.items():
        fitted = clone(model).fit(
            X_train_local,
            y_train_local,
        )

        pred = fitted.predict(X_val_local)

        rows.append(
            {
                "model": name,
                **score_predictions(
                    y_val_local,
                    pred,
                ),
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values(
            ["macro_f1", "balanced_accuracy"],
            ascending=False,
        )
        .reset_index(drop=True)
    )

In [17]:
candidates = make_candidates(RANDOM_STATE)

validation_results = evaluate_candidates(
    train_df,
    val_df,
    candidates,
)

display(validation_results)

selected_model_name = str(
    validation_results.loc[0, "model"]
)

print("The selected model:", selected_model_name)

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,DecisionTree,1.000000,1.000000,1.000000,1.000000
1,RandomForest,0.995799,0.997217,0.997199,0.995799
2,ExtraTrees,0.991593,0.994429,0.994395,0.991593
3,LogisticRegression,0.999875,0.999917,0.923000,0.999877
4,Dummy-most-frequent,0.503079,0.333333,0.223133,0.336760


The selected model: DecisionTree


In [18]:
train_val_df = split_df[
    split_df["split"].isin(
        ["train", "validation"]
    )
].copy()

evaluation_model = clone(
    candidates[selected_model_name]
).fit(
    train_val_df[FEATURES],
    train_val_df["attack_label"],
)

test_pred = evaluation_model.predict(
    test_df[FEATURES]
)

test_metrics = score_predictions(
    test_df["attack_label"],
    test_pred,
)

display(pd.DataFrame([test_metrics]))

,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,1.0,1.0,1.0,1.0


In [19]:
labels = sorted(
    raw_df["attack_label"].unique()
)

report = classification_report(
    test_df["attack_label"],
    test_pred,
    labels=labels,
    target_names=labels,
    output_dict=True,
    zero_division=0,
)

report_df = pd.DataFrame(report).T
display(report_df)

,precision,recall,f1-score,support
dos_udp_flood,0.619048,0.325000,0.426230,40.000000
grey_hole,0.257143,0.642857,0.367347,14.000000
normal,0.910569,0.896000,0.903226,125.000000
accuracy,0.748603,0.748603,0.748603,0.748603
macro avg,0.595587,0.621286,0.565601,179.000000
weighted avg,0.794319,0.748603,0.754722,179.000000


In [20]:
cm = confusion_matrix(
    test_df["attack_label"],
    test_pred,
    labels=labels,
)

cm_df = pd.DataFrame(
    cm,
    index=[f"Thực tế: {label}" for label in labels],
    columns=[f"Dự đoán: {label}" for label in labels],
)

display(cm_df)

NameError: name 'labels' is not defined

In [21]:
perm = permutation_importance(
    evaluation_model,
    test_df[FEATURES],
    test_df["attack_label"],
    scoring="f1_macro",
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=1,
)

importance_df = (
    pd.DataFrame(
        {
            "feature": FEATURES,
            "importance_mean": perm.importances_mean,
            "importance_std": perm.importances_std,
        }
    )
    .sort_values(
        "importance_mean",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(importance_df)

,feature,importance_mean,importance_std
0,delay_ms,0.161739,0.042765
1,jitter_ms,0.093989,0.044847
2,speed,0.014769,0.014625
3,rx_mean_pkt_bytes,0.002524,0.006156
4,rx_pps,0.001591,0.005375
5,rx_bps,0.001457,0.005430
6,window_pdr,0.001103,0.004877
7,drop_pps,0.000995,0.002986
8,tx_bps,0.000542,0.003288
9,tx_pps,-0.000800,0.005664


In [37]:
def run_ablation(
    train_data: pd.DataFrame,
    validation_data: pd.DataFrame,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:

    feature_sets = {
        "traffic_rate": [
            "tx_pps",
            "rx_pps",
            "drop_pps",
            "tx_bps",
            "rx_bps",
            "tx_mean_pkt_bytes",
            "rx_mean_pkt_bytes",
        ],

        "traffic_plus_quality": [
            "tx_pps",
            "rx_pps",
            "drop_pps",
            "tx_bps",
            "rx_bps",
            "tx_mean_pkt_bytes",
            "rx_mean_pkt_bytes",
            "window_pdr",
            "window_plr",
            "delay_ms",
            "jitter_ms",
        ],

        "traffic_quality_mobility": FEATURES,
    }

    rows = []

    for name, columns in feature_sets.items():
        model = Pipeline(
            [
                (
                    "imputer",
                    SimpleImputer(strategy="median"),
                ),
                (
                    "model",
                    RandomForestClassifier(
                        n_estimators=500000,
                        max_depth=12,
                        min_samples_leaf=3,
                        max_features="sqrt",
                        class_weight="balanced_subsample",
                        n_jobs=1,
                        random_state=random_state,
                    ),
                ),
            ]
        )

        model.fit(
            train_data[columns],
            train_data["attack_label"],
        )

        pred = model.predict(
            validation_data[columns]
        )

        rows.append(
            {
                "feature_set": name,
                "n_features": len(columns),
                **score_predictions(
                    validation_data["attack_label"],
                    pred,
                ),
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values(
            "macro_f1",
            ascending=False,
        )
    )

In [23]:
ablation_results = run_ablation(
    train_df,
    val_df,
    RANDOM_STATE,
)

display(ablation_results)

,feature_set,n_features,accuracy,balanced_accuracy,macro_f1,weighted_f1
1,traffic_plus_quality,11,0.751553,0.475758,0.456765,0.718358
2,traffic_quality_mobility,13,0.701863,0.388357,0.385754,0.669400
0,traffic_rate,7,0.285714,0.346093,0.182987,0.202881


In [24]:
def run_tree_count_study(
    train_data: pd.DataFrame,
    validation_data: pd.DataFrame,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:

    rows = []

    for n_trees in [50, 100, 200, 500, 1000]:
        model = Pipeline(
            [
                (
                    "imputer",
                    SimpleImputer(strategy="median"),
                ),
                (
                    "model",
                    RandomForestClassifier(
                        n_estimators=n_trees,
                        max_depth=12,
                        min_samples_leaf=3,
                        max_features="sqrt",
                        class_weight="balanced_subsample",
                        n_jobs=1,
                        random_state=random_state,
                    ),
                ),
            ]
        )

        model.fit(
            train_data[FEATURES],
            train_data["attack_label"],
        )

        pred = model.predict(
            validation_data[FEATURES]
        )

        rows.append(
            {
                "n_estimators": n_trees,
                **score_predictions(
                    validation_data["attack_label"],
                    pred,
                ),
            }
        )

    return pd.DataFrame(rows)

In [25]:
tree_count_results = run_tree_count_study(
    train_df,
    val_df,
    RANDOM_STATE,
)

display(tree_count_results)

,n_estimators,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,50,0.701863,0.388357,0.388268,0.674553
1,100,0.701863,0.388357,0.385754,0.669400
2,200,0.701863,0.388357,0.385754,0.669400
3,500,0.701863,0.388357,0.385754,0.669400
4,1000,0.701863,0.388357,0.385754,0.669400


In [26]:
limitations = [
    "Dataset chỉ có 13 flow.",
    "Grey-hole chỉ có 1 flow.",
    "Chưa có run_id/seed độc lập.",
    "Nhãn tấn công bị kéo dài ở nhiều cửa sổ không có traffic mới.",
    "Temporal split trong cùng flow chỉ là giải pháp tạm thời.",
    "Model là baseline/provisional, không phải kết quả công bố cuối.",
]

for item in limitations:
    print("-", item)

- Dataset chỉ có 13 flow.
- Grey-hole chỉ có 1 flow.
- Chưa có run_id/seed độc lập.
- Nhãn tấn công bị kéo dài ở nhiều cửa sổ không có traffic mới.
- Temporal split trong cùng flow chỉ là giải pháp tạm thời.
- Model là baseline/provisional, không phải kết quả công bố cuối.


In [27]:
evaluation_bundle = {
    "model": evaluation_model,
    "features": FEATURES,
    "feature_engineering_version": "window_delta_v1",
    "selected_model": selected_model_name,
    "labels": labels,
    "test_metrics": test_metrics,
    "leakage_columns_excluded": LEAKAGE_COLUMNS_EXCLUDED,
    "limitations": limitations,
    "training_scope": "train_plus_validation_only",
}

joblib.dump(
    evaluation_bundle,
    OUTPUT_DIR / "fanet_ids_evaluation_bundle.joblib",
)

print("Đã lưu evaluation bundle.")

Đã lưu evaluation bundle.


In [28]:
deployment_model = clone(
    candidates[selected_model_name]
).fit(
    window_df[FEATURES],
    window_df["attack_label"],
)

deployment_bundle = {
    **evaluation_bundle,
    "model": deployment_model,
    "training_scope": "all_available_rows_for_demo",
}

joblib.dump(
    deployment_bundle,
    OUTPUT_DIR / "fanet_ids_demo_bundle.joblib",
)

print("Đã lưu demo bundle.")

Đã lưu demo bundle.


In [29]:
window_df.to_csv(
    OUTPUT_DIR / "tl8_window_features_revised.csv",
    index=False,
)

validation_results.to_csv(
    OUTPUT_DIR / "validation_model_comparison.csv",
    index=False,
)

importance_df.to_csv(
    OUTPUT_DIR / "permutation_importance.csv",
    index=False,
)

ablation_results.to_csv(
    OUTPUT_DIR / "ablation_results.csv",
    index=False,
)

tree_count_results.to_csv(
    OUTPUT_DIR / "tree_count_results.csv",
    index=False,
)

split_counts.to_csv(
    OUTPUT_DIR / "temporal_split_counts.csv",
)

print("Đã lưu các file CSV kết quả.")

Đã lưu các file CSV kết quả.


In [30]:
metrics = {
    "selected_model": selected_model_name,
    "audit": audit,
    "split_counts": (
        split_counts
        .reset_index()
        .to_dict(orient="records")
    ),
    "validation_results": (
        validation_results
        .to_dict(orient="records")
    ),
    "test_metrics": test_metrics,
    "classification_report": report,
    "confusion_matrix": {
        "labels": labels,
        "matrix": cm.tolist(),
    },
    "permutation_importance": (
        importance_df
        .to_dict(orient="records")
    ),
    "ablation_results": (
        ablation_results
        .to_dict(orient="records")
    ),
    "tree_count_results": (
        tree_count_results
        .to_dict(orient="records")
    ),
    "limitations": limitations,
}

metrics_path = (
    OUTPUT_DIR
    / "fanet_ids_revised_metrics.json"
)

with metrics_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        json_safe(metrics),
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Đã lưu:", metrics_path.resolve())

Đã lưu: F:\NCKH_ICTU_2025\Using-machine-learning-to-detect-attacks-in-a-network-of-unmanned-aerial-vehicles-FANET\fanet_nckh_output\fanet_ids_revised_metrics.json


In [31]:
print("Selected model:", selected_model_name)
print("\nTest metrics:")

for key, value in test_metrics.items():
    print(f"  {key}: {value:.4f}")

print("\nOutput directory:")
print(OUTPUT_DIR.resolve())

Selected model: ExtraTrees

Test metrics:
  accuracy: 0.7486
  balanced_accuracy: 0.6213
  macro_f1: 0.5656
  weighted_f1: 0.7547

Output directory:
F:\NCKH_ICTU_2025\Using-machine-learning-to-detect-attacks-in-a-network-of-unmanned-aerial-vehicles-FANET\fanet_nckh_output
